# Topic Modeling: BERTopic vs. FASTopic

To discover the most common topics talked about by customers, this notebook implements and compares two topic modeling approaches on Yelp review text:

1. **Load and prepare the review data**, merging reviews with business names and inspecting the review-count distribution per business.
2. **Fit BERTopic per business** — BERTopic creates contextual review embeddings using a Transformer model, reduces their dimensionality (UMAP), groups semantically related reviews into clusters (HDBSCAN), and uses class-based TF-IDF to identify the words that best represent each topic.
3. **Fit FASTopic per business** — FASTopic is a neural topic model that uses pretrained Transformer embeddings and optimal transport to learn relationships among documents, topics, and words, designed for efficient, stable, and interpretable topic discovery.
4. **Evaluate both models** using C_v coherence, NPMI coherence, U_Mass coherence, and topic diversity, computed per business and aggregated.
5. **Visualize the comparison** across all processed businesses.
6. **Export results** (ranked topics, representative keywords, and topic-frequency data) for use in the Shiny dashboard.

Businesses below a minimum review-count threshold are skipped, since too few reviews cannot produce a statistically meaningful topic cluster. Results are checkpointed per business as individual `.pkl` files, so the pipeline can be safely stopped and resumed without losing completed work.

## 1. Environment Setup: CUDA-Enabled PyTorch and RAPIDS cuML

Both BERTopic (via UMAP + HDBSCAN) and the embedding step benefit heavily from GPU acceleration. This section:
- Reinstalls PyTorch with a CUDA build matching the local GPU/driver (the default `pip install torch` can silently install a CPU-only wheel).
- Confirms RAPIDS cuML's GPU-accelerated `UMAP`/`HDBSCAN` are importable, which can speed up BERTopic's clustering step by an order of magnitude over the CPU-only versions.
- Verifies CUDA is actually available and enables TF32 matrix multiplication precision, which gives a free speedup on modern GPUs (Ampere/Blackwell and later) with negligible accuracy loss.
- Installs `bertopic` and `fastopic`.

In [ ]:
!pip uninstall torch torchvision torchaudio -y

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

In [ ]:
import cuml
print(cuml.__version__)

from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN
print("cuML imports working")

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

In [ ]:
!pip install bertopic fastopic umap-learn hdbscan --quiet

## 2. Imports

Core libraries for data handling, embeddings, both topic models, coherence evaluation (gensim), and thread-pool based concurrent processing (used later to keep the GPU busy across many small per-business FASTopic fits).

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import pickle
import time
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer,ENGLISH_STOP_WORDS
from bertopic import BERTopic
from fastopic import FASTopic
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN
from concurrent.futures import ThreadPoolExecutor, as_completed

# GPU-accelerated implementations (RAPIDS cuML) — used when available and enabled below.
try:
    from cuml.manifold import UMAP as UMAP_GPU
    from cuml.cluster import HDBSCAN as HDBSCAN_GPU
except ImportError:
    UMAP_GPU, HDBSCAN_GPU = None, None

# CPU implementations — always available, used as the non-accelerated fallback.
from umap import UMAP as UMAP_CPU
from hdbscan import HDBSCAN as HDBSCAN_CPU

## 3. Load and Merge Review Data

Loads the cleaned review CSV and merges in each business's name from the business metadata file, joining on `business_id`.

In [ ]:
df = pd.read_csv("data/yelp_reviews_clean_CA.csv")
bn_df = pd.read_csv("data/yelp_academic_dataset_business.csv", low_memory=False)

df = df.merge(
    bn_df[["business_id", "name"]],
    on="business_id",
    how="left"
)
print(f"Total rows: {len(df)}")
print(f"Rows with no matching name: {df['name'].isna().sum()}")
df.head()

In [ ]:
# DONT RUN THIS ONE, HOLDS THE ~7 MIL ROWS CSV
chunks = []
for chunk in pd.read_csv("yelp_reviews_clean.csv", chunksize=5000):
     chunks.append(chunk)
df = pd.concat(chunks, ignore_index=True)

bn_df = pd.read_csv("data/yelp_academic_dataset_business.csv")

df = df.merge(
    bn_df[["business_id", "name"]],
    on="business_id",
    how="left"
)
print(f"Total rows: {len(df)}")
print(f"Rows with no matching name: {df['name'].isna().sum()}")
df.head()

## 4. Business Review-Count Distribution

Topic modeling needs enough documents per business to find real structure — a business with only a handful of reviews can't be meaningfully clustered. This cell checks how many reviews each business has on record, which informs the minimum-review threshold set below.

In [ ]:
business_sizes = df.groupby("business_id").size()
print(f"Total unique businesses: {len(business_sizes)}")
print(business_sizes.describe())
print(f"\nBusinesses with 1 review: {(business_sizes == 1).sum()}")
print(f"Businesses with 10+ reviews: {(business_sizes >= 10).sum()}")

## 5. Embedding Model and Preprocessing Configuration

Loads the sentence-embedding model shared by both BERTopic and the coherence calculations, and defines:
- `MIN_REVIEWS_FOR_TOPIC_MODELING` — businesses below this review count are skipped rather than modeled.
- A custom stopword list (standard English stopwords plus review-specific filler words like *got*, *went*, *asked*) so that topic keywords reflect actual subject matter rather than generic filler.
- `safe_filename` — sanitizes business IDs before using them as checkpoint filenames, since names/IDs can contain filesystem-unsafe characters.
- `clean_for_fastopic` — strips the custom stopwords from review text before it's passed to FASTopic, which (unlike BERTopic) doesn't accept a custom vectorizer for this.

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
MIN_REVIEWS_FOR_TOPIC_MODELING = 15

review_filler_words = {"got", "went", "really", "just", "said", "definitely",
                       "came", "asked", "told", "did", "didn't", "don't",
                       "a", "able", "about", "across", "after", "all", "almost",
                       "also", "am" ,"among", "an", "and", "any", "are", "as",
                       "at", "be", "because", "been", "but", "by", "can", "cannot",
                       "could", "dear", "do", "does", "either", "else", "ever", "every",
                       "for", "from", "get", "got", "had", "has", "have", "he", "her", "hers",
                       "him", "his", "how", "however", "i", "if", "in", "into", "is", "it",
                       "its", "just", "least", "let", "like", "likely", "may", "me",
                       "might", "most", "must", "my", "neither", "no", "nor", "not",
                       "of", "off", "often", "on", "only", "or", "other", "our", "own",
                       "rather", "say", "says", "she", "should", "since", "so", "some",
                       "than", "that", "the", "their", "them", "then", "there", "these",
                       "they", "this", "tis", "to", "too", "twas", "us", "wants", "was",
                       "we", "were", "what", "when", "where", "which", "while", "who",
                       "whom", "why", "will", "with", "would", "yet", "you", "your"}
custom_stopwords = list(ENGLISH_STOP_WORDS.union(review_filler_words))

def safe_filename(s):
    return re.sub(r'[\\/*?:"<>|]', "_", str(s))

def clean_for_fastopic(text, extra_stopwords):
    words = text.lower().split()
    words = [w for w in words if w not in extra_stopwords]
    return " ".join(words)

def compute_all_metrics(topic_words_list, tokenized_docs, top_n_diversity=10):
    if len(topic_words_list) == 0:
        return {
            "coherence_cv": None,
            "coherence_npmi": None,
            "coherence_umass": None,
            "diversity": None,
        }
    dictionary = Dictionary(tokenized_docs)
    corpus = [dictionary.doc2bow(d) for d in tokenized_docs]

    metrics = {}
    for name, coherence_type in [("coherence_cv", "c_v"),
                                   ("coherence_npmi", "c_npmi"),
                                   ("coherence_umass", "u_mass")]:
        cm = CoherenceModel(
            topics=topic_words_list,
            texts=tokenized_docs,
            corpus=corpus,
            dictionary=dictionary,
            coherence=coherence_type
        )
        metrics[name] = cm.get_coherence()

    all_words = []
    for topic in topic_words_list:
        all_words.extend(topic[:top_n_diversity])
    metrics["diversity"] = len(set(all_words)) / len(all_words) if all_words else 0

    return metrics

business_groups = df.groupby("business_id")
total_businesses = business_groups.ngroups
print(f"Total businesses to process (BERTopic): {total_businesses}\n")

### GPU vs. CPU Acceleration Toggle

BERTopic's UMAP (dimensionality reduction) and HDBSCAN (clustering) steps can run on either RAPIDS cuML (GPU-accelerated, often 10-50x faster) or their standard CPU implementations (`umap-learn` / `hdbscan`). Set `USE_GPU_ACCELERATION` below to choose. If GPU acceleration is requested but cuML isn't available in this environment (e.g. no RAPIDS install, or running on a machine without a supported GPU), the notebook automatically falls back to the CPU versions rather than failing.

In [ ]:
USE_GPU_ACCELERATION = True  # Set to False to force CPU-only UMAP/HDBSCAN

if USE_GPU_ACCELERATION and UMAP_GPU is not None:
    UMAP_ACTIVE, HDBSCAN_ACTIVE = UMAP_GPU, HDBSCAN_GPU
    print("Using GPU-accelerated UMAP/HDBSCAN (RAPIDS cuML)")
elif USE_GPU_ACCELERATION and UMAP_GPU is None:
    UMAP_ACTIVE, HDBSCAN_ACTIVE = UMAP_CPU, HDBSCAN_CPU
    print("USE_GPU_ACCELERATION=True but cuML is not available — falling back to CPU UMAP/HDBSCAN")
else:
    UMAP_ACTIVE, HDBSCAN_ACTIVE = UMAP_CPU, HDBSCAN_CPU
    print("Using CPU UMAP/HDBSCAN")

## 6. BERTopic: Per-Business Topic Modeling

Fits a separate BERTopic model for each business that meets the minimum review-count threshold. For every business, this:
- Encodes its reviews with the shared embedding model.
- Fits BERTopic with stopword-filtered, bigram-aware vectorization and a review-count-scaled `min_topic_size` (so small businesses aren't held to the same clustering bar as large ones).
- Computes C_v, NPMI, and U_Mass coherence plus topic diversity.
- Saves each business's result as its own checkpointed `.pkl` file, so the run can be stopped and resumed without recomputation, and failures for one business don't block the rest.

In [ ]:
bertopic_results_dir = "bertopic_results"
os.makedirs(bertopic_results_dir, exist_ok=True)
bertopic_failed = []

umap_model = UMAP_ACTIVE(n_neighbors=15, n_components=5, min_dist=0.0)
hdbscan_model = HDBSCAN_ACTIVE(min_cluster_size=10)

vectorizer_model = CountVectorizer(stop_words="english", min_df=1, ngram_range=(1, 2))

start_all = time.time()

for biz_num, (biz_id, group) in enumerate(business_groups, start=1):
    result_path = f"{bertopic_results_dir}/{safe_filename(biz_id)}.pkl"

    if os.path.exists(result_path):
        print(f"[{biz_num}/{total_businesses}] {biz_id}: already done, skipping")
        continue

    biz_docs = group["text"].astype(str).tolist()
    biz_name = group["name"].iloc[0]

    if len(biz_docs) < MIN_REVIEWS_FOR_TOPIC_MODELING:
        print(f"[{biz_num}/{total_businesses}] {biz_id} ({biz_name}): only {len(biz_docs)} reviews, skipping (below minimum)")
        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id, "business_name": biz_name,
                "num_reviews": len(biz_docs), "skipped": True, "reason": "insufficient_reviews"
            }, f)
        continue

    print(f"[{biz_num}/{total_businesses}] {biz_id} ({biz_name}): {len(biz_docs)} reviews")

    try:
        fit_start = time.time()

        biz_embeddings = embedding_model.encode(biz_docs, batch_size=96, show_progress_bar=False)

        min_topic_size = max(2, min(10, len(biz_docs) // 5))

        bertopic_model = BERTopic(
            embedding_model=embedding_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            calculate_probabilities=False,
            verbose=False,
            min_topic_size=min_topic_size
        )
        bertopic_topics, _ = bertopic_model.fit_transform(biz_docs, biz_embeddings)

        fit_seconds = time.time() - fit_start

        topic_info = bertopic_model.get_topics()
        bertopic_topic_words = [[w for w, _ in words] for tid, words in topic_info.items() if tid != -1]

        topic_counts = Counter([int(t) for t in bertopic_topics])
        topic_counts.pop(-1, None)

        tokenized = [d.lower().split() for d in biz_docs]
        metrics = compute_all_metrics(bertopic_topic_words, tokenized)

        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id, "business_name": biz_name, "num_reviews": len(biz_docs),
                "topics": bertopic_topics, "topic_words": bertopic_topic_words,
                "topic_counts": dict(topic_counts), "fit_seconds": fit_seconds,
                **metrics
            }, f)

    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
        bertopic_failed.append((biz_id, biz_name, len(biz_docs), str(e)))
        continue

elapsed = time.time() - start_all
print(f"\nBERTopic done. {len(bertopic_failed)} failed out of {total_businesses}. Took {elapsed/60:.1f} min")

Businesses that failed during BERTopic fitting, for later inspection:

In [ ]:
bertopic_failures_df = pd.DataFrame(bertopic_failed, columns=["business_id", "business_name", "num_reviews", "error"])
bertopic_failures_df.sort_values("num_reviews").head(20)

## 7. FASTopic: Per-Business Topic Modeling

Mirrors the BERTopic loop above, fitting a separate FASTopic model per business. FASTopic's topic count is matched to whatever BERTopic found for that same business (when available) for a fair, comparable topic count between the two models. Processing runs in small concurrent batches via a thread pool to keep the GPU utilized across many independent, lightweight per-business fits.

In [ ]:
fastopic_results_dir = "fastopic_results"
os.makedirs(fastopic_results_dir, exist_ok=True)

def process_one_business(biz_docs_cleaned, num_topics_guess):
    fit_start = time.time()
    model = FASTopic(num_topics=num_topics_guess, verbose=False, device='cuda')
    topic_words, doc_topics = model.fit_transform(biz_docs_cleaned, learning_rate=0.01, epochs=50)
    fit_seconds = time.time() - fit_start
    return topic_words, doc_topics, fit_seconds

fastopic_failed = []

start_all = time.time()

BATCH_SIZE = 10  # matches max_workers below — tune based on your GPU headroom
batch = []  # holds (biz_id, biz_name, biz_docs, biz_docs_cleaned, num_topics_guess) for businesses that need processing

def flush_batch(batch):
    """Process one batch concurrently, save results, log failures."""
    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        future_to_biz = {
            executor.submit(process_one_business, item[3], item[4]): item
            for item in batch
        }
        for future in as_completed(future_to_biz):
            biz_id, biz_name, biz_docs, biz_docs_cleaned, num_topics_guess = future_to_biz[future]
            result_path = f"{fastopic_results_dir}/{safe_filename(biz_id)}.pkl"
            try:
                fastopic_topic_words, fastopic_doc_topics, fit_seconds = future.result()

                fastopic_topic_words = [
                    topic.split() if isinstance(topic, str) else list(topic)
                    for topic in fastopic_topic_words
                ]

                doc_topics_array = np.array(fastopic_doc_topics)
                assigned_topics = doc_topics_array.argmax(axis=1) if doc_topics_array.ndim > 1 else doc_topics_array
                topic_counts = Counter([int(t) for t in assigned_topics])

                tokenized = [d.lower().split() for d in biz_docs]
                metrics = compute_all_metrics(fastopic_topic_words, tokenized)

                with open(result_path, "wb") as f:
                    pickle.dump({
                        "business_id": biz_id, "business_name": biz_name, "num_reviews": len(biz_docs),
                        "topic_words": fastopic_topic_words, "doc_topics": fastopic_doc_topics,
                        "assigned_topics": assigned_topics.tolist(),
                        "topic_counts": dict(topic_counts), "fit_seconds": fit_seconds,
                        **metrics
                    }, f)
                print(f"  done: {biz_id} ({biz_name})")

            except Exception as e:
                print(f"  FAILED: {biz_id} ({biz_name}) — {type(e).__name__}: {e}")
                fastopic_failed.append((biz_id, biz_name, len(biz_docs), str(e)))

for biz_num, (biz_id, group) in enumerate(business_groups, start=1):
    result_path = f"{fastopic_results_dir}/{safe_filename(biz_id)}.pkl"

    if os.path.exists(result_path):
        continue  # already done, skip silently (no need to print every skip when batching)

    biz_docs = group["text"].astype(str).tolist()
    biz_name = group["name"].iloc[0]

    if len(biz_docs) < MIN_REVIEWS_FOR_TOPIC_MODELING:
        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id, "business_name": biz_name,
                "num_reviews": len(biz_docs), "skipped": True, "reason": "insufficient_reviews"
            }, f)
        continue

    bertopic_path = f"{bertopic_results_dir}/{safe_filename(biz_id)}.pkl"
    if os.path.exists(bertopic_path):
        with open(bertopic_path, "rb") as f:
            bertopic_result = pickle.load(f)
        num_topics_guess = 5 if bertopic_result.get("skipped", False) else max(len(bertopic_result["topic_words"]), 2)
    else:
        num_topics_guess = 5

    biz_docs_cleaned = [clean_for_fastopic(d, custom_stopwords) for d in biz_docs]
    batch.append((biz_id, biz_name, biz_docs, biz_docs_cleaned, num_topics_guess))

    if len(batch) >= BATCH_SIZE:
        print(f"[~{biz_num}/{total_businesses}] processing batch of {len(batch)}...")
        flush_batch(batch)
        batch = []

# process any leftover partial batch at the end
if batch:
    print(f"processing final batch of {len(batch)}...")
    flush_batch(batch)

elapsed = time.time() - start_all
print(f"\nFASTopic done. {len(fastopic_failed)} failed out of {total_businesses}. Took {elapsed/60:.1f} min")

Businesses that failed during FASTopic fitting:

In [ ]:
fastopic_failures_df = pd.DataFrame(fastopic_failed, columns=["business_id", "business_name", "num_reviews", "error"])
fastopic_failures_df.sort_values("num_reviews").head(20)

Helper to recover each review's star rating and align it with its assigned topic, used later to compute `average_topic_score` per topic without needing to refit any models.

In [ ]:
def compute_average_topic_scores(results_dir, model_name, df, topics_key):
    """Returns {(model, business_id, topic_id): avg_star_rating}."""
    scores = {}

    for fname in os.listdir(results_dir):
        if not fname.endswith(".pkl"):
            continue

        with open(os.path.join(results_dir, fname), "rb") as f:
            result = pickle.load(f)

        if result.get("skipped", False):
            continue

        biz_id = result.get("business_id")
        assigned = result.get(topics_key)  # "topics" for BERTopic, "assigned_topics" for FASTopic
        if assigned is None:
            continue

        group = df[df["business_id"] == biz_id]
        if len(group) != len(assigned):
            continue  # safety check: skip if row counts don't line up

        stars = group["stars"].tolist()

        topic_stars = {}
        for topic_id, star in zip(assigned, stars):
            if topic_id == -1:  # BERTopic outlier topic, skip
                continue
            topic_stars.setdefault(topic_id, []).append(star)

        for topic_id, star_list in topic_stars.items():
            scores[(model_name, biz_id, topic_id)] = float(np.mean(star_list))

    return scores


bertopic_scores = compute_average_topic_scores(bertopic_results_dir, "BERTopic", df, "topics")
fastopic_scores = compute_average_topic_scores(fastopic_results_dir, "FASTopic", df, "assigned_topics")
all_topic_scores = {**bertopic_scores, **fastopic_scores}

Utility to inspect the discovered topics for a single business side by side across both models — useful for sanity-checking topic quality before trusting the aggregate metrics below.

In [ ]:
def show_topics(biz_id, top_n_words=8):
    bertopic_path = f"{bertopic_results_dir}/{biz_id}.pkl"
    fastopic_path = f"{fastopic_results_dir}/{biz_id}.pkl"

    if not os.path.exists(bertopic_path) or not os.path.exists(fastopic_path):
        print(f"No results found for {biz_id} in one or both folders.")
        return

    with open(bertopic_path, "rb") as f:
        b = pickle.load(f)
    with open(fastopic_path, "rb") as f:
        fst = pickle.load(f)

    if b.get("skipped") or fst.get("skipped"):
        print(f"{biz_id} was skipped (insufficient reviews).")
        return

    print(f"=== {b.get('business_name')} ({biz_id}) — {b.get('num_reviews')} reviews ===\n")

    print(f"BERTopic ({len(b['topic_words'])} topics):")
    for i, words in enumerate(b["topic_words"]):
        print(f"  Topic {i}: {', '.join(words[:top_n_words])}")

    print(f"\nFASTopic ({len(fst['topic_words'])} topics):")
    for i, words in enumerate(fst["topic_words"]):
        print(f"  Topic {i}: {', '.join(words[:top_n_words])}")

# Example usage — pick any business_id you know has finished processing
show_topics("nUqrF-h9S7myCcvNDecOvw")

## 8. Evaluation Metrics: Coherence and Diversity

Builds a per-business comparison table from the checkpointed results of both models:
- **C_v coherence** — correlates well with human judgment of topic quality; the standard metric reported in most topic-modeling papers.
- **NPMI coherence** — a faster alternative that also aligns reasonably well with human judgment.
- **U_Mass coherence** — the fastest to compute (no external reference corpus needed), useful as a sanity check.
- **Topic diversity** — the fraction of unique words across all of a business's topics; measures whether topics are actually distinct from one another, rather than all restating the same idea.

Only businesses successfully processed by *both* models are included, so the comparison is apples-to-apples.

In [ ]:
comparison_rows = []

for fname in os.listdir(bertopic_results_dir):
    if not fname.endswith(".pkl"):
        continue  # skip .ipynb_checkpoints and any other non-result files

    biz_id = fname.replace(".pkl", "")
    bertopic_path = f"{bertopic_results_dir}/{fname}"
    fastopic_path = f"{fastopic_results_dir}/{fname}"

    if not os.path.exists(fastopic_path):
        continue

    with open(bertopic_path, "rb") as f:
        b = pickle.load(f)
    with open(fastopic_path, "rb") as f:
        fst = pickle.load(f)

    comparison_rows.append({
        "business_id": biz_id,
        "business_name": b.get("business_name"),
        "num_reviews": b.get("num_reviews"),
        "skipped": b.get("skipped", False),

        "bertopic_coherence_cv": b.get("coherence_cv"),
        "bertopic_coherence_npmi": b.get("coherence_npmi"),
        "bertopic_coherence_umass": b.get("coherence_umass"),
        "bertopic_diversity": b.get("diversity"),

        "fastopic_coherence_cv": fst.get("coherence_cv"),
        "fastopic_coherence_npmi": fst.get("coherence_npmi"),
        "fastopic_coherence_umass": fst.get("coherence_umass"),
        "fastopic_diversity": fst.get("diversity"),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df_valid = comparison_df[~comparison_df["skipped"]].copy()
comparison_df

## 9. Visualization

The following cells chart the model comparison from several angles: average performance, score distributions, whether business size affects topic quality, how often each model "wins" head-to-head, and what the actual discovered topics look like for a representative business.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

### 9.1 Mean Coherence and Diversity Comparison

Average score per metric, BERTopic vs. FASTopic, across all processed businesses.

In [ ]:
metrics_config = [
    ("coherence_cv", "C_V Coherence"),
    ("coherence_npmi", "NPMI Coherence"),
    ("coherence_umass", "U_Mass Coherence"),
    ("diversity", "Topic Diversity"),
]

for metric_key, title in metrics_config:
    bertopic_mean = comparison_df_valid[f"bertopic_{metric_key}"].mean()
    fastopic_mean = comparison_df_valid[f"fastopic_{metric_key}"].mean()

    fig, ax = plt.subplots(figsize=(6, 5))
    bars = ax.bar(["BERTopic", "FASTopic"], [bertopic_mean, fastopic_mean],
                   color=["#4C72B0", "#DD8452"])
    ax.set_ylabel(title)
    ax.set_title(f"Average {title}: BERTopic vs FASTopic")

    # label each bar with its value
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.4f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom")

    plt.tight_layout()
    plt.savefig(f"mean_{metric_key}_comparison.png", dpi=150)
    plt.show()

### 9.2 Distribution of Coherence Scores

Box plots showing the spread of each coherence metric, not just its mean — reveals whether one model is more consistent than the other.

In [ ]:
for metric_key, title in metrics_config[:3]:  # only the 3 coherence metrics have meaningful distributions to compare this way
    fig, ax = plt.subplots(figsize=(6, 5))
    data = [comparison_df_valid[f"bertopic_{metric_key}"].dropna(),
            comparison_df_valid[f"fastopic_{metric_key}"].dropna()]
    ax.boxplot(data, tick_labels=["BERTopic", "FASTopic"])
    ax.set_ylabel(title)
    ax.set_title(f"{title} Distribution: BERTopic vs FASTopic")
    plt.tight_layout()
    plt.savefig(f"distribution_{metric_key}.png", dpi=150)
    plt.show()

### 9.3 Coherence vs. Business Size

Checks whether businesses with more reviews tend to produce more coherent topics, for either model.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(comparison_df_valid["num_reviews"], comparison_df_valid["bertopic_coherence_cv"],
           alpha=0.4, label="BERTopic", s=15)
ax.scatter(comparison_df_valid["num_reviews"], comparison_df_valid["fastopic_coherence_cv"],
           alpha=0.4, label="FASTopic", s=15)
ax.set_xscale("log")
ax.set_xlabel("Number of Reviews (log scale)")
ax.set_ylabel("C_V Coherence")
ax.set_title("Coherence vs. Business Review Count")
ax.legend()
plt.tight_layout()
plt.savefig("coherence_vs_size.png", dpi=150)
plt.show()

### 9.4 Win Rate

For each business, which model produced the higher C_v coherence score — summarized as a head-to-head win rate.

In [ ]:
comparison_df_valid["bertopic_wins"] = (
    comparison_df_valid["bertopic_coherence_cv"] > comparison_df_valid["fastopic_coherence_cv"]
)
win_counts = comparison_df_valid["bertopic_wins"].value_counts()

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    [win_counts.get(True, 0), win_counts.get(False, 0)],
    labels=["BERTopic higher C_V", "FASTopic higher C_V"],
    autopct="%1.1f%%",
    colors=["#4C72B0", "#DD8452"]
)
ax.set_title("Which Model Produces Higher Coherence, Per Business")
plt.tight_layout()
plt.savefig("win_rate_pie.png", dpi=150)
plt.show()

### 9.5 Ranked Topic Bars for a Selected Business

A concrete, human-readable example: the actual topics and their relative frequency for one business, shown for both models.

In [ ]:
def plot_topic_bars(biz_id, results_dir, model_name, top_n_topics=10):
    with open(f"{results_dir}/{biz_id}.pkl", "rb") as f:
        result = pickle.load(f)

    topic_words = result["topic_words"]
    topic_counts = result.get("topic_counts", {})

    if not topic_counts:
        print(f"No topic_counts found for {biz_id} — rerun this business to regenerate with counts.")
        return

    # Sort topics by frequency, descending
    sorted_topics = sorted(topic_counts.items(), key=lambda x: x[1], reverse=True)[:top_n_topics]

    labels = []
    sizes = []
    for topic_id, count in sorted_topics:
        idx = topic_id if topic_id >= 0 else 0  # guard for BERTopic's -1 indexing quirks
        words = topic_words[idx][:3] if idx < len(topic_words) else ["?"]
        labels.append(", ".join(words))
        sizes.append(count)

    fig, ax = plt.subplots(figsize=(8, max(3, len(labels) * 0.4)))
    ax.barh(labels[::-1], sizes[::-1], color="#55A868")  # reverse so highest is on top
    ax.set_xlabel("Number of Reviews")
    ax.set_title(f"{model_name} Topics — {result.get('business_name', biz_id)}")
    plt.tight_layout()
    plt.show()

# Example usage:
example_biz_id = comparison_df_valid.iloc[0]["business_id"]
plot_topic_bars(example_biz_id, bertopic_results_dir, "BERTopic")
plot_topic_bars(example_biz_id, fastopic_results_dir, "FASTopic")

## 10. Export for the Shiny Dashboard

Produces the flat CSV files the dashboard reads at startup:
- `exports/business_top_topics.csv` — one row per (business, topic, model), used to render ranked topics and frequency charts per business.
- `exports/bertopic_topics.csv` — one row per (topic, model), holding the representative keywords shown in the dashboard.
- `exports/topic_model_comparison.csv` — one row per model, holding the aggregate metrics shown on the Models tab.

In [ ]:
def build_topic_exports(results_dir, model_name, topic_scores):
    business_rows = []
    topic_label_rows = []

    total_docs = 0
    total_outliers = 0
    topic_count_per_business = []
    fit_seconds_list = []

    for fname in os.listdir(results_dir):
        if not fname.endswith(".pkl"):
            continue

        with open(os.path.join(results_dir, fname), "rb") as f:
            result = pickle.load(f)

        if result.get("skipped", False):
            continue

        business_name = result.get("business_name")
        biz_id = result.get("business_id")
        num_reviews = result.get("num_reviews", 0)
        topic_words = result.get("topic_words", [])
        topic_counts = result.get("topic_counts", {})

        total_docs += num_reviews
        topic_count_per_business.append(len(topic_words))

        if "fit_seconds" in result:
            fit_seconds_list.append(result["fit_seconds"])

        if model_name.casefold() == "bertopic":
            assigned = result.get("topics", [])
            outliers = sum(1 for t in assigned if t == -1)
            total_outliers += outliers

        sorted_topics = sorted(topic_counts.items(), key=lambda x: x[1], reverse=True)
        total_assigned = sum(count for _, count in sorted_topics) or 1

        for rank, (topic_id, count) in enumerate(sorted_topics, start=1):
            words = topic_words[topic_id][:5] if topic_id < len(topic_words) else []
            topic_label = ", ".join(words[:3]) if words else f"Topic {topic_id}"

            business_rows.append({
                "business_name": business_name,
                "topic_id": topic_id,
                "topic_label": topic_label,
                "topic_review_count": count,
                "topic_share": count / total_assigned,
                "topic_rank": rank,
                "model": model_name,
                "average_topic_score": topic_scores.get((model_name, biz_id, topic_id), np.nan),  # NEW
            })

            topic_label_rows.append({
                "topic_id": topic_id,
                "topic_label": topic_label,
                "top_words": ", ".join(words),
                "model": model_name,
            })

    outlier_rate = (total_outliers / total_docs) if (model_name.casefold() == "bertopic" and total_docs) else np.nan
    mean_num_topics = float(np.mean(topic_count_per_business)) if topic_count_per_business else np.nan
    mean_fit_seconds = float(np.mean(fit_seconds_list)) if fit_seconds_list else np.nan  # NEW

    return business_rows, topic_label_rows, total_docs, mean_num_topics, outlier_rate, mean_fit_seconds


all_business_rows = []
all_topic_label_rows = []
model_summary_rows = []

for model_name, results_dir in [("BERTopic", bertopic_results_dir), ("FASTopic", fastopic_results_dir)]:
    business_rows, topic_label_rows, total_docs, mean_num_topics, outlier_rate, mean_fit_seconds = build_topic_exports(
        results_dir, model_name, all_topic_scores
    )
    all_business_rows.extend(business_rows)
    all_topic_label_rows.extend(topic_label_rows)

    prefix = "bertopic" if model_name.casefold() == "bertopic" else "fastopic"
    model_summary_rows.append({
        "model": model_name,
        "fit_documents": total_docs,
        "number_of_topics": mean_num_topics,
        "c_v": comparison_df_valid[f"{prefix}_coherence_cv"].mean(),
        "c_npmi": comparison_df_valid[f"{prefix}_coherence_npmi"].mean(),
        "u_mass": comparison_df_valid[f"{prefix}_coherence_umass"].mean(),
        "topic_diversity": comparison_df_valid[f"{prefix}_diversity"].mean(),
        "fit_seconds": mean_fit_seconds,  # NEW
        "outlier_rate": outlier_rate,
    })

business_top_topics_df = pd.DataFrame(all_business_rows)
bertopic_topics_df = pd.DataFrame(all_topic_label_rows)
topic_model_comparison_df = pd.DataFrame(model_summary_rows)

business_top_topics_df.to_csv("exports/business_top_topics.csv", index=False)
bertopic_topics_df.to_csv("exports/bertopic_topics.csv", index=False)
topic_model_comparison_df.to_csv("exports/topic_model_comparison.csv", index=False)

print(f"business_top_topics.csv: {len(business_top_topics_df)} rows, "
      f"{business_top_topics_df['average_topic_score'].notna().sum()} with a score")
print(f"topic_model_comparison.csv:\n{topic_model_comparison_df}")